In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [8]:
import os
import copy
import json
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    log_loss,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

import mlflow
import mlflow.pytorch
from mlflow.models.signature import infer_signature
import dagshub

DATASET_ROOT = Path("/kaggle/input/datasets/sowmyabarla/parkinsons-augmented-handwriting-dataset/augmented_hw_dataset")
MODALITY_FOLDER = "spiral"
DATA_DIR = DATASET_ROOT / MODALITY_FOLDER

OUTPUT_DIR = Path("/kaggle/working/artifacts")
PLOTS_DIR = OUTPUT_DIR / "plots"
MODELS_DIR = OUTPUT_DIR / "models"
EMBEDDINGS_DIR = OUTPUT_DIR / "embeddings"
RUN_ID_FILE = OUTPUT_DIR / "mlflow_run_id.txt"

from kaggle_secrets import UserSecretsClient

def get_secret_or_env(key: str, default: str = "") -> str:
    try:
        return UserSecretsClient().get_secret(key)
    except Exception:
        return os.getenv(key, default)

DAGSHUB_ORG_NAME = get_secret_or_env("DAGSHUB_ORG_NAME", "YOUR_ORG_NAME")
DAGSHUB_REPO_NAME = get_secret_or_env("DAGSHUB_REPO_NAME", "YOUR_REPO_NAME")
DAGSHUB_USERNAME = get_secret_or_env("DAGSHUB_USERNAME", "YOUR_PERSONAL_USERNAME")
DAGSHUB_TOKEN = get_secret_or_env("DAGSHUB_TOKEN", "YOUR_PERSONAL_TOKEN")

EXPERIMENT_NAME = "Parkinsons_Multimodal_Pipeline"
RUN_NAME = "Spiral_Image_Experiment"
REUSE_SHARED_RUN_ID = True

BASE_MODEL = "mobilenet_v3_small"
PRETRAINED = True
FREEZE_RATIO = 0.5
EMBEDDING_DIM = 128
DROPOUT_RATE = 0.4

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 35
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
EARLY_STOPPING_PATIENCE = 8
CLASSIFICATION_THRESHOLD = 0.5
SEED = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
CLASS_MAPPING = {"healthy": 0, "parkinson": 1}
INV_CLASS_MAPPING = {0: "healthy", 1: "parkinson"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [44]:
from mlflow.tracking import MlflowClient
from mlflow.tracking.fluent import _active_run_stack

def set_global_seed(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False








def setup_directories():
    for directory in [OUTPUT_DIR, PLOTS_DIR, MODELS_DIR, EMBEDDINGS_DIR]:
        directory.mkdir(parents=True, exist_ok=True)


def setup_mlflow():
    if (
        DAGSHUB_ORG_NAME != "YOUR_ORG_NAME"
        and DAGSHUB_REPO_NAME != "YOUR_REPO_NAME"
        and DAGSHUB_TOKEN != "YOUR_PERSONAL_TOKEN"
    ):
        tracking_uri = f"https://dagshub.com/{DAGSHUB_ORG_NAME}/{DAGSHUB_REPO_NAME}.mlflow"
        os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
        os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
        os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

        dagshub.auth.add_app_token(DAGSHUB_TOKEN)
        dagshub.init(
            repo_owner=DAGSHUB_ORG_NAME,
            repo_name=DAGSHUB_REPO_NAME,
            mlflow=True,
        )
        mlflow.set_tracking_uri(tracking_uri)
    else:
        mlflow.set_tracking_uri(f"file://{OUTPUT_DIR / 'mlruns'}")

    mlflow.set_experiment(EXPERIMENT_NAME)



def clear_active_mlflow_runs():
    while mlflow.active_run() is not None:
        try:
            mlflow.end_run()
        except Exception:
            if len(_active_run_stack.get()) > 0:
                _active_run_stack.get().pop()


def start_or_resume_run(force_new: bool = False):
    setup_directories()
    clear_active_mlflow_runs()
    client = MlflowClient()

    if REUSE_SHARED_RUN_ID and not force_new and RUN_ID_FILE.exists():
        existing_run_id = RUN_ID_FILE.read_text().strip()
        if existing_run_id:
            try:
                client.update_run(existing_run_id, status="RUNNING")
                return mlflow.start_run(run_id=existing_run_id)
            except Exception:
                pass

    run = mlflow.start_run()
    RUN_ID_FILE.write_text(run.info.run_id)
    return run


set_global_seed(SEED)
setup_directories()
setup_mlflow()
setup_directories()
clear_active_mlflow_runs()

Initialized MLflow to track repo "Parkinson/multiModal"

Repository Parkinson/multiModal initialized!

In [38]:
def build_split_dataframe(split_dir: Path) -> pd.DataFrame:
    valid_extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif"}
    records = []
    for class_name, label in CLASS_MAPPING.items():
        class_dir = split_dir / class_name
        if not class_dir.exists():
            continue
        for file_path in sorted(class_dir.rglob("*")):
            if file_path.suffix.lower() in valid_extensions:
                records.append({
                    "filepath": str(file_path),
                    "filename": file_path.name,
                    "class_name": class_name,
                    "label": label,
                })
    return pd.DataFrame(records)


class SpiralHandwritingDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index: int):
        row = self.dataframe.iloc[index]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row["label"], dtype=torch.float32)
        return image, label, row["filename"]


train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomAutocontrast(p=0.4),
    transforms.RandomRotation(degrees=15, fill=255),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_df = build_split_dataframe(DATA_DIR / "training")
val_df = build_split_dataframe(DATA_DIR / "validation")
test_df = build_split_dataframe(DATA_DIR / "testing")

train_dataset = SpiralHandwritingDataset(train_df, transform=train_transforms)
val_dataset = SpiralHandwritingDataset(val_df, transform=eval_transforms)
test_dataset = SpiralHandwritingDataset(test_df, transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Testing: {len(test_df)}")

Train: 288 | Validation: 72 | Testing: 150


In [14]:
class ConfigurableVisionClassifier(nn.Module):
    def __init__(
        self,
        base_model_name: str = BASE_MODEL,
        pretrained: bool = PRETRAINED,
        embedding_dim: int = EMBEDDING_DIM,
        dropout_rate: float = DROPOUT_RATE,
        freeze_ratio: float = FREEZE_RATIO,
    ):
        super().__init__()
        self.base_model_name = base_model_name.lower()
        self.backbone, in_features = self._build_backbone(self.base_model_name, pretrained)
        self._apply_freezing(freeze_ratio)

        self.embedding_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.Hardswish(inplace=True),
            nn.Dropout(p=dropout_rate),
        )
        self.classifier = nn.Linear(embedding_dim, 1)

    def _build_backbone(self, name: str, pretrained: bool):
        if name == "mobilenet_v3_small":
            weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
            net = models.mobilenet_v3_small(weights=weights)
            backbone = nn.Sequential(net.features, net.avgpool)
            in_features = net.classifier[0].in_features
        elif name == "mobilenet_v3_large":
            weights = models.MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
            net = models.mobilenet_v3_large(weights=weights)
            backbone = nn.Sequential(net.features, net.avgpool)
            in_features = net.classifier[0].in_features
        elif name == "mobilenet_v2":
            weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
            net = models.mobilenet_v2(weights=weights)
            backbone = nn.Sequential(net.features, nn.AdaptiveAvgPool2d((1, 1)))
            in_features = net.classifier[1].in_features
        elif name == "resnet18":
            weights = models.ResNet18_Weights.DEFAULT if pretrained else None
            net = models.resnet18(weights=weights)
            backbone = nn.Sequential(*list(net.children())[:-1])
            in_features = net.fc.in_features
        elif name == "resnet34":
            weights = models.ResNet34_Weights.DEFAULT if pretrained else None
            net = models.resnet34(weights=weights)
            backbone = nn.Sequential(*list(net.children())[:-1])
            in_features = net.fc.in_features
        elif name == "efficientnet_b0":
            weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
            net = models.efficientnet_b0(weights=weights)
            backbone = nn.Sequential(net.features, net.avgpool)
            in_features = net.classifier[1].in_features
        elif name == "densenet121":
            weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
            net = models.densenet121(weights=weights)
            backbone = nn.Sequential(net.features, nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d((1, 1)))
            in_features = net.classifier.in_features
        else:
            raise ValueError(f"Unsupported BASE_MODEL: {name}")
        return backbone, in_features

    def _apply_freezing(self, freeze_ratio: float):
        if freeze_ratio <= 0.0:
            return
        children = list(self.backbone[0].children()) if len(list(self.backbone.children())) > 0 else list(self.backbone.children())
        freeze_count = int(len(children) * freeze_ratio)
        for layer in children[:freeze_count]:
            for param in layer.parameters():
                param.requires_grad = False

    def extract_embedding(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.embedding_layer(features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedding = self.extract_embedding(x)
        return self.classifier(embedding).squeeze(-1)

In [15]:
def calculate_all_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = CLASSIFICATION_THRESHOLD) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    clipped_prob = np.clip(y_prob, 1e-7, 1 - 1e-7)

    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        "recall_sensitivity": round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        "specificity": round(float(specificity), 4),
        "npv": round(float(npv), 4),
        "f1_score": round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
        "roc_auc": round(float(roc_auc_score(y_true, y_prob)), 4),
        "pr_auc": round(float(average_precision_score(y_true, y_prob)), 4),
        "mcc": round(float(matthews_corrcoef(y_true, y_pred)), 4),
        "log_loss": round(float(log_loss(y_true, clipped_prob, labels=[0, 1])), 4),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }


def save_loss_and_auc_plot(history: dict, save_path: Path):
    epochs_range = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_range, history["train_loss"], label="Train Loss", linewidth=2)
    axes[0].plot(epochs_range, history["val_loss"], label="Validation Loss", linewidth=2)
    axes[0].set_title(f"{BASE_MODEL} Loss Curve")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("BCE Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_range, history["val_roc_auc"], label="Val ROC-AUC", linewidth=2)
    axes[1].plot(epochs_range, history["val_pr_auc"], label="Val PR-AUC", linewidth=2)
    axes[1].plot(epochs_range, history["val_f1"], label="Val F1-Score", linewidth=2)
    axes[1].set_title(f"{BASE_MODEL} Validation Metrics Curve")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def save_roc_curve_plot(y_true: np.ndarray, y_prob: np.ndarray, roc_auc: float, save_path: Path):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, linewidth=2.5, label=f"{BASE_MODEL} (ROC-AUC = {roc_auc:.4f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Chance (0.5000)")
    ax.set_title(f"ROC Curve - {MODALITY_FOLDER.capitalize()} ({BASE_MODEL})")
    ax.set_xlabel("False Positive Rate (1 - Specificity)")
    ax.set_ylabel("True Positive Rate (Sensitivity)")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def save_precision_recall_plot(y_true: np.ndarray, y_prob: np.ndarray, pr_auc: float, save_path: Path):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    baseline = float(np.mean(y_true))
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(recall, precision, linewidth=2.5, color="darkorange", label=f"{BASE_MODEL} (PR-AUC = {pr_auc:.4f})")
    ax.axhline(y=baseline, linestyle="--", color="gray", label=f"Baseline ({baseline:.4f})")
    ax.set_title(f"Precision-Recall Curve - {MODALITY_FOLDER.capitalize()} ({BASE_MODEL})")
    ax.set_xlabel("Recall (Sensitivity)")
    ax.set_ylabel("Precision")
    ax.legend(loc="lower left")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def save_confusion_matrix_plot(y_true: np.ndarray, y_prob: np.ndarray, threshold: float, save_path: Path):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Healthy (0)", "Parkinson (1)"],
        yticklabels=["Healthy (0)", "Parkinson (1)"],
        ax=ax,
    )
    ax.set_title(f"Confusion Matrix - {BASE_MODEL}")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

In [46]:
import gc
import shutil
from itertools import product

MIN_F1_TO_SAVE_MODEL = 0.87
SEARCH_MODE = "sampled_per_model"
RUNS_PER_MODEL = 4

MODEL_SEARCH_SPACE = {
    "base_model": [
        "mobilenet_v3_small",
        "mobilenet_v3_large",
        "mobilenet_v2",
        "resnet18",
        "resnet34",
        "efficientnet_b0",
        "densenet121",
    ],
    "lr": [5e-4, 3e-4, 1e-4],
    "weight_decay": [1e-2, 1e-3, 1e-4],
    "freeze_ratio": [0.0, 0.3, 0.5],
    "dropout_rate": [0.2, 0.3, 0.5],
    "embedding_dim": [64, 128, 256],
    "batch_size": [16, 32],
    "optimizer": ["adamw", "adam", "radam"],
    "scheduler": ["cosine", "plateau", "step"],
    "label_smoothing": [0.0, 0.05],
}


def build_experiment_configs(search_space: dict, mode: str = "sampled_per_model", runs_per_model: int = 4, seed: int = 42) -> list:
    rng = random.Random(seed)
    other_keys = [k for k in search_space.keys() if k != "base_model"]
    other_combinations = [dict(zip(other_keys, vals)) for vals in product(*[search_space[k] for k in other_keys])]

    configs = []
    for model_name in search_space["base_model"]:
        if mode == "full_grid" or len(other_combinations) <= runs_per_model:
            selected = other_combinations
        else:
            selected = rng.sample(other_combinations, runs_per_model)
        for combo in selected:
            configs.append({"base_model": model_name, **combo})
    return configs


EXPERIMENT_CONFIGS = build_experiment_configs(
    MODEL_SEARCH_SPACE,
    mode=SEARCH_MODE,
    runs_per_model=RUNS_PER_MODEL,
    seed=SEED,
)


def build_optimizer(model: nn.Module, cfg: dict):
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    opt_name = cfg["optimizer"].lower()
    if opt_name == "adamw":
        return optim.AdamW(trainable_params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    if opt_name == "adam":
        return optim.Adam(trainable_params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    if opt_name == "radam":
        return optim.RAdam(trainable_params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    if opt_name == "sgd":
        return optim.SGD(trainable_params, lr=cfg["lr"], momentum=0.9, weight_decay=cfg["weight_decay"], nesterov=True)
    raise ValueError(f"Unsupported optimizer: {opt_name}")


def build_scheduler(optimizer, cfg: dict):
    sched_name = cfg["scheduler"].lower()
    if sched_name == "cosine":
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    if sched_name == "plateau":
        return optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6)
    if sched_name == "step":
        return optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)
    raise ValueError(f"Unsupported scheduler: {sched_name}")


def find_optimal_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    best_thresh = 0.50
    best_f1 = -1.0
    for thresh in np.linspace(0.25, 0.75, 51):
        preds = (y_prob >= thresh).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresh = float(thresh)
    return round(best_thresh, 4)


def run_evaluation(model: nn.Module, loader: DataLoader, criterion: nn.Module):
    model.eval()
    running_loss = 0.0
    probs_list, labels_list, embeddings_list, filenames_list = [], [], [], []

    with torch.no_grad():
        for images, labels, filenames in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            embeddings = model.extract_embedding(images)
            logits = model.classifier(embeddings).squeeze(-1)
            loss = criterion(logits, labels)
            probs = torch.sigmoid(logits)

            running_loss += loss.item() * images.size(0)
            probs_list.extend(probs.cpu().numpy().tolist())
            labels_list.extend(labels.cpu().numpy().tolist())
            embeddings_list.append(embeddings.cpu().numpy())
            filenames_list.extend(filenames)

    epoch_loss = running_loss / len(loader.dataset)
    return (
        epoch_loss,
        np.array(labels_list, dtype=np.int64),
        np.array(probs_list, dtype=np.float32),
        np.vstack(embeddings_list),
        filenames_list,
    )


def train_and_evaluate_single_config(exp_idx: int, cfg: dict, pos_weight_tensor: torch.Tensor) -> dict:
    global BASE_MODEL
    BASE_MODEL = cfg["base_model"]
    exp_tag = (
        f"exp{exp_idx:02d}_{cfg['base_model']}_"
        f"{cfg['optimizer']}_lr{cfg['lr']}_bs{cfg['batch_size']}_fr{cfg['freeze_ratio']}"
    )

    set_global_seed(SEED)

    exp_train_loader = DataLoader(train_dataset, batch_size=cfg["batch_size"], shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    exp_val_loader = DataLoader(val_dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    exp_test_loader = DataLoader(test_dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = ConfigurableVisionClassifier(
        base_model_name=cfg["base_model"],
        pretrained=PRETRAINED,
        embedding_dim=cfg["embedding_dim"],
        dropout_rate=cfg["dropout_rate"],
        freeze_ratio=cfg["freeze_ratio"],
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg)

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_roc_auc": [],
        "val_pr_auc": [],
        "val_f1": [],
    }

    best_composite_score = -float("inf")
    best_model_state = copy.deepcopy(model.state_dict())
    patience_counter = 0

    with mlflow.start_run(run_name=exp_tag, nested=True):
        mlflow.log_params({
            "exp_tag": exp_tag,
            "img_modality": MODALITY_FOLDER,
            "img_base_model": cfg["base_model"],
            "img_pretrained": PRETRAINED,
            "img_lr": cfg["lr"],
            "img_weight_decay": cfg["weight_decay"],
            "img_freeze_ratio": cfg["freeze_ratio"],
            "img_dropout_rate": cfg["dropout_rate"],
            "img_embedding_dim": cfg["embedding_dim"],
            "img_batch_size": cfg["batch_size"],
            "img_optimizer": cfg["optimizer"],
            "img_scheduler": cfg["scheduler"],
            "img_label_smoothing": cfg["label_smoothing"],
            "img_size": IMAGE_SIZE,
            "img_epochs": EPOCHS,
            "img_min_f1_to_save": MIN_F1_TO_SAVE_MODEL,
        })

        for epoch in range(1, EPOCHS + 1):
            model.train()
            running_train_loss = 0.0

            for images, labels, _ in exp_train_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)

                if cfg["label_smoothing"] > 0.0:
                    smoothed_labels = labels * (1.0 - cfg["label_smoothing"]) + 0.5 * cfg["label_smoothing"]
                else:
                    smoothed_labels = labels

                optimizer.zero_grad()
                logits = model(images)
                loss = criterion(logits, smoothed_labels)
                loss.backward()
                optimizer.step()

                running_train_loss += loss.item() * images.size(0)

            train_loss = running_train_loss / len(exp_train_loader.dataset)
            val_loss, y_val, p_val, _, _ = run_evaluation(model, exp_val_loader, criterion)
            val_metrics = calculate_all_metrics(y_val, p_val, threshold=CLASSIFICATION_THRESHOLD)

            if cfg["scheduler"] == "plateau":
                scheduler.step(val_loss)
            else:
                scheduler.step()

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_roc_auc"].append(val_metrics["roc_auc"])
            history["val_pr_auc"].append(val_metrics["pr_auc"])
            history["val_f1"].append(val_metrics["f1_score"])

            mlflow.log_metrics({
                "img_train_loss": round(train_loss, 4),
                "img_val_loss": round(val_loss, 4),
                "img_val_roc_auc": val_metrics["roc_auc"],
                "img_val_pr_auc": val_metrics["pr_auc"],
                "img_val_f1": val_metrics["f1_score"],
                "img_val_accuracy": val_metrics["accuracy"],
            }, step=epoch)

            composite_score = val_metrics["f1_score"] + val_metrics["roc_auc"] - (0.25 * val_loss)

            if composite_score > best_composite_score:
                best_composite_score = composite_score
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= EARLY_STOPPING_PATIENCE:
                break

        model.load_state_dict(best_model_state)
        model.eval()

        _, y_train_eval, p_train_eval, emb_train, files_train = run_evaluation(
            model,
            DataLoader(SpiralHandwritingDataset(train_df, transform=eval_transforms), batch_size=cfg["batch_size"], shuffle=False),
            criterion,
        )
        _, y_val_eval, p_val_eval, emb_val, files_val = run_evaluation(model, exp_val_loader, criterion)
        test_loss, y_test, p_test, emb_test, files_test = run_evaluation(model, exp_test_loader, criterion)

        optimal_threshold = find_optimal_threshold(y_val_eval, p_val_eval)
        val_final_metrics = calculate_all_metrics(y_val_eval, p_val_eval, threshold=optimal_threshold)
        test_metrics = calculate_all_metrics(y_test, p_test, threshold=optimal_threshold)
        test_metrics["test_bce_loss"] = round(float(test_loss), 4)

        loss_plot_path = PLOTS_DIR / f"{exp_tag}_loss_and_metrics_curve.png"
        roc_plot_path = PLOTS_DIR / f"{exp_tag}_roc_curve.png"
        pr_plot_path = PLOTS_DIR / f"{exp_tag}_precision_recall_curve.png"
        cm_plot_path = PLOTS_DIR / f"{exp_tag}_confusion_matrix.png"

        save_loss_and_auc_plot(history, loss_plot_path)
        save_roc_curve_plot(y_test, p_test, test_metrics["roc_auc"], roc_plot_path)
        save_precision_recall_plot(y_test, p_test, test_metrics["pr_auc"], pr_plot_path)
        save_confusion_matrix_plot(y_test, p_test, optimal_threshold, cm_plot_path)

        metrics_json_path = OUTPUT_DIR / f"{exp_tag}_metrics.json"
        with open(metrics_json_path, "w") as f:
            json.dump({"config": cfg, "optimal_threshold": optimal_threshold, "validation": val_final_metrics, "testing": test_metrics}, f, indent=4)

        qualifies_for_saving = test_metrics["f1_score"] > MIN_F1_TO_SAVE_MODEL
        mlflow.set_tag("img_model_saved", str(qualifies_for_saving))
        mlflow.log_metric("img_optimal_threshold", optimal_threshold)

        for metric_name, metric_value in val_final_metrics.items():
            mlflow.log_metric(f"img_val_final_{metric_name}", metric_value)
        for metric_name, metric_value in test_metrics.items():
            mlflow.log_metric(f"img_test_{metric_name}", metric_value)

        for plot_file in [loss_plot_path, roc_plot_path, pr_plot_path, cm_plot_path]:
            mlflow.log_artifact(str(plot_file), artifact_path=f"plots/{exp_tag}")

        mlflow.log_artifact(str(metrics_json_path), artifact_path=f"metrics/{exp_tag}")

        saved_pickle_path = None

        if qualifies_for_saving:
            sample_input_tensor = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
            with torch.no_grad():
                sample_logit = model(sample_input_tensor)
                sample_prob = torch.sigmoid(sample_logit).cpu().numpy()

            sample_input_np = sample_input_tensor.cpu().numpy()
            mlflow_signature = infer_signature(sample_input_np, sample_prob)

            model_schema = {
                "model_name": f"spiral_{exp_tag}_classifier",
                "modality": MODALITY_FOLDER,
                "base_model": cfg["base_model"],
                "hyperparameters": cfg,
                "input_schema": {
                    "tensor_shape": [1, 3, IMAGE_SIZE, IMAGE_SIZE],
                    "dtype": "float32",
                    "color_space": "RGB",
                    "normalization_mean": IMAGENET_MEAN,
                    "normalization_std": IMAGENET_STD,
                },
                "output_schema": {
                    "logit_shape": [1],
                    "probability_range": [0.0, 1.0],
                    "embedding_shape": [1, cfg["embedding_dim"]],
                    "threshold": optimal_threshold,
                    "class_mapping": CLASS_MAPPING,
                    "inverse_class_mapping": INV_CLASS_MAPPING,
                },
                "mlflow_signature": mlflow_signature.to_dict(),
            }

            model_cpu = copy.deepcopy(model).cpu()
            pickle_bundle = {
                "model": model_cpu,
                "state_dict": model_cpu.state_dict(),
                "schema": model_schema,
                "val_metrics": val_final_metrics,
                "test_metrics": test_metrics,
            }

            saved_pickle_path = MODELS_DIR / f"spiral_{exp_tag}_with_schema.pkl"
            with open(saved_pickle_path, "wb") as f:
                pickle.dump(pickle_bundle, f)

            embeddings_bundle_path = EMBEDDINGS_DIR / f"spiral_{exp_tag}_embeddings.pkl"
            with open(embeddings_bundle_path, "wb") as f:
                pickle.dump({
                    "train": {"labels": y_train_eval, "probs": p_train_eval, "embeddings": emb_train, "filenames": files_train},
                    "validation": {"labels": y_val_eval, "probs": p_val_eval, "embeddings": emb_val, "filenames": files_val},
                    "testing": {"labels": y_test, "probs": p_test, "embeddings": emb_test, "filenames": files_test},
                }, f)

            local_mlflow_model_dir = MODELS_DIR / f"mlflow_pytorch_{exp_tag}"
            if local_mlflow_model_dir.exists():
                shutil.rmtree(local_mlflow_model_dir)

            mlflow.pytorch.save_model(
                pytorch_model=model_cpu,
                path=str(local_mlflow_model_dir),
                signature=mlflow_signature,
                input_example=sample_input_np,
            )

            mlflow.log_artifact(str(saved_pickle_path), artifact_path=f"models/{exp_tag}/pickle_bundle")
            mlflow.log_artifact(str(embeddings_bundle_path), artifact_path=f"embeddings/{exp_tag}")
            mlflow.log_artifacts(str(local_mlflow_model_dir), artifact_path=f"models/{exp_tag}/pytorch_model")

        print(
            f"[{exp_idx:02d}/{len(EXPERIMENT_CONFIGS):02d}] {exp_tag} | "
            f"Thresh: {optimal_threshold:.2f} | "
            f"Val F1: {val_final_metrics['f1_score']:.4f} | "
            f"Test F1: {test_metrics['f1_score']:.4f} | "
            f"Test AUC: {test_metrics['roc_auc']:.4f} | "
            f"Saved: {qualifies_for_saving}"
        )

    del model, optimizer, scheduler, criterion
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "exp_tag": exp_tag,
        **cfg,
        "optimal_threshold": optimal_threshold,
        "val_f1_score": val_final_metrics["f1_score"],
        "val_roc_auc": val_final_metrics["roc_auc"],
        **{f"test_{k}": v for k, v in test_metrics.items()},
        "model_saved": qualifies_for_saving,
        "pickle_path": str(saved_pickle_path) if saved_pickle_path else None,
    }


In [47]:
pos_count = int((train_df["label"] == 1).sum())
neg_count = int((train_df["label"] == 0).sum())
pos_weight_tensor = torch.tensor([neg_count / max(pos_count, 1)], dtype=torch.float32, device=DEVICE)

all_experiment_results = []

with start_or_resume_run() as parent_run:
    for idx, exp_config in enumerate(EXPERIMENT_CONFIGS, start=1):
        result_row = train_and_evaluate_single_config(idx, exp_config, pos_weight_tensor)
        all_experiment_results.append(result_row)

    leaderboard_df = pd.DataFrame(all_experiment_results).sort_values(
        by=["test_f1_score", "test_roc_auc", "test_mcc"],
        ascending=False,
    ).reset_index(drop=True)

    leaderboard_csv_path = OUTPUT_DIR / "spiral_models_leaderboard.csv"
    leaderboard_df.to_csv(leaderboard_csv_path, index=False)
    mlflow.log_artifact(str(leaderboard_csv_path), artifact_path="leaderboard")

    best_row = leaderboard_df.iloc[0].to_dict()
    mlflow.set_tags({
        "best_img_exp_tag": str(best_row["exp_tag"]),
        "best_img_base_model": str(best_row["base_model"]),
        "best_img_lr": str(best_row["lr"]),
        "best_img_freeze_ratio": str(best_row["freeze_ratio"]),
    })
    mlflow.log_metrics({
        "best_img_test_f1_score": float(best_row["test_f1_score"]),
        "best_img_test_roc_auc": float(best_row["test_roc_auc"]),
        "best_img_test_pr_auc": float(best_row["test_pr_auc"]),
        "best_img_test_accuracy": float(best_row["test_accuracy"]),
        "best_img_test_recall_sensitivity": float(best_row["test_recall_sensitivity"]),
        "best_img_test_specificity": float(best_row["test_specificity"]),
        "best_img_test_mcc": float(best_row["test_mcc"]),
    })

display(leaderboard_df)

[01/28] exp01_mobilenet_v3_small_adam_lr0.0005_bs32_fr0.5 | Thresh: 0.26 | Val F1: 0.9254 | Test F1: 0.7681 | Test AUC: 0.8558 | Saved: False
🏃 View run exp01_mobilenet_v3_small_adam_lr0.0005_bs32_fr0.5 at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0/runs/74720088835741a4b8197a8b5a0ea475
🧪 View experiment at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0
[02/28] exp02_mobilenet_v3_small_radam_lr0.0005_bs16_fr0.3 | Thresh: 0.44 | Val F1: 0.9444 | Test F1: 0.7160 | Test AUC: 0.8178 | Saved: False
🏃 View run exp02_mobilenet_v3_small_radam_lr0.0005_bs16_fr0.3 at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0/runs/edb26ac477dc4655aa0f8416a848d044
🧪 View experiment at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0


W1111 21:59:15.606000 49 torch/_export/non_strict_utils.py:646] dimension inputs['x'].shape[0] 0/1 specialized; Dim.AUTO was specified along with a sample input with hint = 1.
2026/11/11 21:59:15 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/11 21:59:28 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /kaggle/working/artifacts/models/mlflow_pytorch_exp03_mobilenet_v3_small_adam_lr0.0003_bs16_fr0.3/data, flavor: pytorch). Fall back to return ['torch==2.11.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


[03/28] exp03_mobilenet_v3_small_adam_lr0.0003_bs16_fr0.3 | Thresh: 0.37 | Val F1: 1.0000 | Test F1: 0.8805 | Test AUC: 0.9493 | Saved: True
🏃 View run exp03_mobilenet_v3_small_adam_lr0.0003_bs16_fr0.3 at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0/runs/244796ce9c784ced9050af3e7028969e
🧪 View experiment at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0
[04/28] exp04_mobilenet_v3_small_radam_lr0.0003_bs16_fr0.0 | Thresh: 0.68 | Val F1: 1.0000 | Test F1: 0.7591 | Test AUC: 0.8770 | Saved: False
🏃 View run exp04_mobilenet_v3_small_radam_lr0.0003_bs16_fr0.0 at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0/runs/3d602292efb44d519a6fa35a069e60c8
🧪 View experiment at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0
[05/28] exp05_mobilenet_v3_large_adamw_lr0.0003_bs32_fr0.5 | Thresh: 0.34 | Val F1: 0.8387 | Test F1: 0.6508 | Test AUC: 0.8020 | Saved: False
🏃 View run exp05_mobilenet_v3_large_adamw_lr0.0003_bs32_fr0.5 at: 

W1111 22:08:16.184000 49 torch/_export/non_strict_utils.py:646] dimension inputs['x'].shape[0] 0/1 specialized; Dim.AUTO was specified along with a sample input with hint = 1.
2026/11/11 22:08:16 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/11 22:08:29 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /kaggle/working/artifacts/models/mlflow_pytorch_exp07_mobilenet_v3_large_adamw_lr0.0005_bs32_fr0.5/data, flavor: pytorch). Fall back to return ['torch==2.11.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


[07/28] exp07_mobilenet_v3_large_adamw_lr0.0005_bs32_fr0.5 | Thresh: 0.25 | Val F1: 1.0000 | Test F1: 0.8947 | Test AUC: 0.9476 | Saved: True
🏃 View run exp07_mobilenet_v3_large_adamw_lr0.0005_bs32_fr0.5 at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0/runs/00f56a358c7d45669f955a18b6fdfb3e
🧪 View experiment at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0
[08/28] exp08_mobilenet_v3_large_adamw_lr0.0005_bs32_fr0.3 | Thresh: 0.25 | Val F1: 0.8286 | Test F1: 0.7413 | Test AUC: 0.8276 | Saved: False
🏃 View run exp08_mobilenet_v3_large_adamw_lr0.0005_bs32_fr0.3 at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0/runs/876dc90375e548439f59b2eea155a9da
🧪 View experiment at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0


W1111 22:12:40.529000 49 torch/_export/non_strict_utils.py:646] dimension inputs['x'].shape[0] 0/1 specialized; Dim.AUTO was specified along with a sample input with hint = 1.
2026/11/11 22:12:40 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/11 22:12:53 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /kaggle/working/artifacts/models/mlflow_pytorch_exp09_mobilenet_v2_adamw_lr0.0001_bs16_fr0.0/data, flavor: pytorch). Fall back to return ['torch==2.11.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


... (20 outputs left out)

[26/28] exp26_densenet121_radam_lr0.0003_bs16_fr0.5 | Thresh: 0.25 | Val F1: 1.0000 | Test F1: 0.8795 | Test AUC: 0.9422 | Saved: True
🏃 View run exp26_densenet121_radam_lr0.0003_bs16_fr0.5 at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0/runs/6b50043cb8e64c14889648015641ff39
🧪 View experiment at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0


W1111 23:05:37.115000 49 torch/_export/non_strict_utils.py:646] dimension inputs['x'].shape[0] 0/1 specialized; Dim.AUTO was specified along with a sample input with hint = 1.
2026/11/11 23:05:38 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/11/11 23:05:51 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /kaggle/working/artifacts/models/mlflow_pytorch_exp27_densenet121_adam_lr0.0005_bs32_fr0.3/data, flavor: pytorch). Fall back to return ['torch==2.11.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


[27/28] exp27_densenet121_adam_lr0.0005_bs32_fr0.3 | Thresh: 0.25 | Val F1: 1.0000 | Test F1: 0.9045 | Test AUC: 0.9618 | Saved: True
🏃 View run exp27_densenet121_adam_lr0.0005_bs32_fr0.3 at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0/runs/a8f5c2d93df74ab0ada42c9c5b04137f
🧪 View experiment at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0
[28/28] exp28_densenet121_radam_lr0.0003_bs32_fr0.3 | Thresh: 0.25 | Val F1: 1.0000 | Test F1: 0.8571 | Test AUC: 0.9335 | Saved: False
🏃 View run exp28_densenet121_radam_lr0.0003_bs32_fr0.3 at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0/runs/bbf743af65084d0ca64122feb1ed1347
🧪 View experiment at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0
🏃 View run youthful-ram-29 at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0/runs/ec68e3af2d814499828420b1cc404930
🧪 View experiment at: https://dagshub.com/Parkinson/multiModal.mlflow/#/experiments/0


,exp_tag,base_model,lr,weight_decay,freeze_ratio,dropout_rate,embedding_dim,batch_size,optimizer,scheduler,...,test_pr_auc,test_mcc,test_log_loss,test_tp,test_tn,test_fp,test_fn,test_test_bce_loss,model_saved,pickle_path
0,exp20_resnet34_radam_lr0.0001_bs16_fr0.3,resnet34,0.0001,0.0010,0.3,0.5,64,16,radam,plateau,...,0.9637,0.8595,0.2897,74,65,10,1,0.2897,True,/kaggle/working/artifacts/models/spiral_exp20_...
1,exp19_resnet34_adam_lr0.0003_bs16_fr0.5,resnet34,0.0003,0.0100,0.5,0.2,128,16,adam,step,...,0.9456,0.8540,0.2497,71,68,7,4,0.2497,True,/kaggle/working/artifacts/models/spiral_exp19_...
2,exp17_resnet34_radam_lr0.0003_bs16_fr0.3,resnet34,0.0003,0.0100,0.3,0.2,64,16,radam,step,...,0.9343,0.8145,0.3093,70,66,9,5,0.3093,True,/kaggle/working/artifacts/models/spiral_exp17_...
3,exp27_densenet121_adam_lr0.0005_bs32_fr0.3,densenet121,0.0005,0.0001,0.3,0.5,128,32,adam,plateau,...,0.9591,0.8035,0.2792,71,64,11,4,0.2792,True,/kaggle/working/artifacts/models/spiral_exp27_...
4,exp12_mobilenet_v2_adamw_lr0.0005_bs32_fr0.3,mobilenet_v2,0.0005,0.0010,0.3,0.5,64,32,adamw,step,...,0.9451,0.7969,0.2847,73,61,14,2,0.2847,True,/kaggle/working/artifacts/models/spiral_exp12_...
5,exp07_mobilenet_v3_large_adamw_lr0.0005_bs32_f...,mobilenet_v3_large,0.0005,0.0010,0.5,0.2,128,32,adamw,step,...,0.9438,0.7869,0.3282,68,66,9,7,0.3282,True,/kaggle/working/artifacts/models/spiral_exp07_...
6,exp18_resnet34_radam_lr0.0001_bs32_fr0.0,resnet34,0.0001,0.0010,0.0,0.2,128,32,radam,plateau,...,0.8711,0.7818,0.3146,72,61,14,3,0.3146,True,/kaggle/working/artifacts/models/spiral_exp18_...
7,exp09_mobilenet_v2_adamw_lr0.0001_bs16_fr0.0,mobilenet_v2,0.0001,0.0010,0.0,0.3,64,16,adamw,cosine,...,0.9561,0.7767,0.2891,70,63,12,5,0.2891,True,/kaggle/working/artifacts/models/spiral_exp09_...
8,exp03_mobilenet_v3_small_adam_lr0.0003_bs16_fr0.3,mobilenet_v3_small,0.0003,0.0010,0.3,0.5,256,16,adam,cosine,...,0.9478,0.7521,0.3075,70,61,14,5,0.3075,True,/kaggle/working/artifacts/models/spiral_exp03_...
9,exp26_densenet121_radam_lr0.0003_bs16_fr0.5,densenet121,0.0003,0.0010,0.5,0.2,64,16,radam,step,...,0.9333,0.7506,0.3288,73,57,18,2,0.3288,True,/kaggle/working/artifacts/models/spiral_exp26_...


In [49]:
def predict_from_pickle(image_path: str, pickle_path: Path = pickle_model_path) -> dict:
    with open(pickle_path, "rb") as f:
        bundle = pickle.load(f)

    schema = bundle["schema"]
    loaded_model = bundle["model"].to(DEVICE)
    loaded_model.eval()

    img_size = schema["input_schema"]["tensor_shape"][2]
    norm_mean = schema["input_schema"]["normalization_mean"]
    norm_std = schema["input_schema"]["normalization_std"]
    threshold = schema["output_schema"]["threshold"]
    inv_map = schema["output_schema"]["inverse_class_mapping"]

    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=norm_mean, std=norm_std),
    ])

    image = Image.open(image_path).convert("RGB")
    tensor_in = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        embedding = loaded_model.extract_embedding(tensor_in)
        logit = loaded_model.classifier(embedding).squeeze(-1)
        probability = float(torch.sigmoid(logit).item())

    predicted_class = int(probability >= threshold)
    return {
        "filepath": image_path,
        "prediction": predicted_class,
        "label": inv_map[predicted_class],
        "probability_pd": round(probability, 4),
        "embedding_shape": list(embedding.shape),
        "embedding": embedding.cpu().numpy().flatten(),
    }





In [55]:
saved_Rows = leaderboard_df[leaderboard_df["model_saved"] == True]

if not saved_Rows.empty:
    best_pickle_model_path = Path(saved_Rows.iloc[0]["pickle_path"])
    sample_test_image = test_df.iloc[145]["filepath"]
    inference_output = predict_from_pickle(sample_test_image, pickle_path=best_pickle_model_path)
    print(f"Loaded Best Qualifying Model: {best_pickle_model_path.name}")
    print({k: v for k, v in inference_output.items() if k != "embedding"})
else:
    print(f"No model exceeded the F1 threshold of {MIN_F1_TO_SAVE_MODEL}")

Loaded Best Qualifying Model: spiral_exp20_resnet34_radam_lr0.0001_bs16_fr0.3_with_schema.pkl
{'filepath': '/kaggle/input/datasets/sowmyabarla/parkinsons-augmented-handwriting-dataset/augmented_hw_dataset/spiral/testing/parkinson/aug_4_V09PE01.png', 'prediction': 1, 'label': 'parkinson', 'probability_pd': 0.8936, 'embedding_shape': [1, 64]}


In [52]:
sample_test_image

'/kaggle/input/datasets/sowmyabarla/parkinsons-augmented-handwriting-dataset/augmented_hw_dataset/spiral/testing/healthy/aug_0_V01HE01.png'

In [56]:
test_df.iloc[145]

filepath      /kaggle/input/datasets/sowmyabarla/parkinsons-...
filename                                      aug_4_V09PE01.png
class_name                                            parkinson
label                                                         1
Name: 145, dtype: object